# Supervised Learning

This notebook uses simple supervised learning to classify images into classes, without data augmentation

In [1]:

cd ..

In [2]:
# install requirements

# if connected to colab


if 'google.colab' in str(get_ipython()):
  print('running on colab')
  %pip install git+https://github.com/henrysky/astroNN.git
else:
  %pip install --upgrade pip
  %pip install -r user-requirements.txt

In [3]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import h5py
import numpy as np
from tensorflow.keras import utils
import os

filepath = 'src/Galaxy10.h5'

if not os.path.exists(filepath):
    print(f"file {filepath} not found")
    print("fetching from http://astro.utoronto.ca/~bovy/Galaxy10/Galaxy10.h5...")
    os.makedirs('src', exist_ok=True)
    !wget -P src http://astro.utoronto.ca/~bovy/Galaxy10/Galaxy10.h5

with h5py.File(filepath, 'r') as F:
    images = np.array(F['images'])
    labels = np.array(F['ans'])

# To convert the labels to categorical 10 classes
labels = utils.to_categorical(labels, 10)

# To convert to desirable type
labels = labels.astype(np.float32)
images = images.astype(np.float32)

In [4]:
images.shape

In [5]:
# divide images and labels into training and test sets

X_train, X_test, y_train, y_test = train_test_split(images, labels, test_size=0.4, random_state=42)

In [6]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

idx = np.random.randint(0, y_train.shape[0], size=10)

for ax, i in zip(axes.flat, idx):
    ax.imshow(X_train[i].astype(np.uint8))
    ax.set_title(f"Class {np.argmax(y_train[i])}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.savefig("plots/original.png")
plt.show()

In [7]:
# def center_crop(img, crop_size):
#     h, w = img.shape[:2]
#     start_h = (h - crop_size) // 2
#     start_w = (w - crop_size) // 2
#     return img[start_h:start_h + crop_size, start_w:start_w + crop_size]

# crop_size = 128
# X_train_cropped = np.array([center_crop(img, crop_size) for img in X_train])
# X_test_cropped = np.array([center_crop(img, crop_size) for img in X_test])

# print(X_train_cropped.shape)

In [8]:
from PIL import Image
import numpy as np

X_train_gray = np.array([
    np.array(Image.fromarray(img.astype(np.uint8)).convert('L'))
    for img in X_train
])

X_test_gray = np.array([
    np.array(Image.fromarray(img.astype(np.uint8)).convert('L'))
    for img in X_test
])

In [9]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for ax, i in zip(axes.flat, idx):
    ax.imshow(X_train_gray[i], cmap="gray")
    ax.set_title(f"Class {y_train[i]}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

### Data Cleaning

1. normalization (min-max)
2. normalization using percentiles
3. gaussian / bilateral filter
4. histogram equalization

In [10]:
# Min-max normalization (per-image)

X_min = X_train_gray.min(axis=(1,2), keepdims=True)  
X_max = X_train_gray.max(axis=(1,2), keepdims=True)
X_norm = (X_train_gray - X_min) / (X_max - X_min + 1e-8)  

In [11]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for ax, i in zip(axes.flat, idx):
    ax.imshow(X_norm[i], cmap="gray")
    ax.set_title(f"Class {y_train[i]}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.savefig("plots/gray.png")
plt.show()

In [12]:
X_norm.shape

In [13]:
# gaussian filter on images

from scipy.ndimage import gaussian_filter

X_gauss = gaussian_filter(X_norm, sigma=1, axes=[1,2])

In [14]:
X_gauss.min(), X_gauss.max()

In [15]:
fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for ax, i in zip(axes.flat, idx):
    ax.imshow(X_gauss[i], cmap="gray")
    ax.set_title(f"Class {y_train[i]}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.savefig("plots/gaussian_smooth.png")
plt.show()

In [16]:
# train a model on X_norm (without filters) and X_gauss

X_norm.shape, y_train.shape

### Training

#### without filter

In [17]:
# train a CNN model

import tensorflow as tf
from keras import layers, models

In [18]:

input_shape = (X_norm.shape[1], X_norm.shape[1], 1)
model = models.Sequential()

model.add(layers.Input(shape=(69, 69, 1)))

# block 1
model.add(layers.Conv2D(8, (3, 3), activation="relu", padding="same"))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2, 2)))

# block 2
model.add(layers.Conv2D(32, (5, 5), activation="relu", padding="same"))
model.add(layers.MaxPooling2D((2, 2)))

# block 3
model.add(layers.Conv2D(64, (3, 3), activation="relu", padding="same"))
model.add(layers.BatchNormalization())
model.add(layers.MaxPooling2D((2, 2)))

# converts feature maps → single vector per feature map
model.add(layers.GlobalAveragePooling2D())

model.add(layers.Dense(32, activation='relu'))

# model.add(layers.Dropout(0.4))

model.add(layers.Dense(10, activation='softmax'))

In [19]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [20]:
X_norm.shape

In [21]:
X_flat = X_norm[..., np.newaxis]
y_labels = np.argmax(y_train, axis=1)


In [22]:
X_flat.shape

In [23]:
y_train.shape

In [24]:
history = model.fit(
    X_flat, y_train,
    epochs=50,
    batch_size=32,
    shuffle=True
)

In [25]:
y = model.predict(X_flat)

In [26]:
y_pred = (y > 0.5).astype(int)
y_pred = np.argmax(y_pred, axis=1)

In [27]:
y_true = np.argmax(y_train, axis=1)

In [28]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

cm = confusion_matrix(y_pred, y_true)

In [29]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", values_format="d")
plt.title("train set - no smooth")
plt.savefig("plots/train_no_smooth.pdf")
plt.show()

#### with filter

In [30]:
X_flat_gauss = X_gauss[..., np.newaxis]

input_shape = (X_norm.shape[1], X_norm.shape[1], 1)
gauss_model = models.Sequential()
gauss_model.add(layers.Input(shape=(69, 69, 1)))

# block 1
gauss_model.add(layers.Conv2D(8, (3, 3), activation="relu", padding="same"))
gauss_model.add(layers.BatchNormalization())
gauss_model.add(layers.MaxPooling2D((2, 2)))

# block 2
gauss_model.add(layers.Conv2D(32, (5, 5), activation="relu", padding="same"))
gauss_model.add(layers.MaxPooling2D((2, 2)))

# block 3
gauss_model.add(layers.Conv2D(64, (3, 3), activation="relu", padding="same"))
gauss_model.add(layers.BatchNormalization())
gauss_model.add(layers.MaxPooling2D((2, 2)))

# converts feature maps → single vector per feature map
gauss_model.add(layers.GlobalAveragePooling2D())

gauss_model.add(layers.Dense(32, activation='relu'))

# model.add(layers.Dropout(0.4))

gauss_model.add(layers.Dense(10, activation='softmax'))

In [31]:
gauss_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [32]:
history = gauss_model.fit(
    X_flat_gauss, y_train,
    epochs=50,
    batch_size=32,
    shuffle=True
)

In [33]:
y_gauss = gauss_model.predict(X_flat_gauss)
y_pred = (y_gauss > 0.5).astype(int)
y_pred = np.argmax(y_gauss, axis=1)

In [34]:
y_true = np.argmax(y_train, axis=1)

In [35]:
cm = confusion_matrix(y_pred, y_true)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", values_format="d")
plt.title("train set - gauss smooth")
plt.savefig("plots/train_gauss_smooth.pdf")
plt.show()

### Testing

#### without filter

In [36]:
X_test_gray.shape

In [37]:
# Min-max normalization (per-image)

X_min = X_test_gray.min(axis=(1,2), keepdims=True)  
X_max = X_test_gray.max(axis=(1,2), keepdims=True)
X_norm = (X_test_gray - X_min) / (X_max - X_min + 1e-8)  

In [38]:
X_flat = X_norm[..., np.newaxis]
y_labels = np.argmax(y_test, axis=1)

In [39]:
y_pred = model.predict(X_flat)
y_pred = (y_pred > 0.5).astype(int)
y_pred = np.argmax(y_pred, axis=1)

In [40]:
y_true = np.argmax(y_test, axis=1)

In [41]:
cm = confusion_matrix(y_pred, y_true)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", values_format="d")
plt.title("Test set - no smooth")
plt.savefig("plots/test_no_smooth.pdf")
plt.show()

In [42]:
X_min = X_test_gray.min(axis=(1,2), keepdims=True)  
X_max = X_test_gray.max(axis=(1,2), keepdims=True)
X_norm = (X_test_gray - X_min) / (X_max - X_min + 1e-8)  

X_gauss = gaussian_filter(X_norm, sigma=1, axes=[1,2])
X_flat = X_gauss[..., np.newaxis]

In [43]:
y_pred = gauss_model.predict(X_flat)
y_pred = (y_pred > 0.5).astype(int)
y_pred = np.argmax(y_pred, axis=1)

with filter

In [44]:
cm = confusion_matrix(y_pred, y_true)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues", values_format="d")
plt.title("Test set - gauss smooth")
plt.savefig("plots/test_gauss_smooth.pdf")
plt.show()

### Network inspect

In [45]:
from tensorflow.keras.models import Model

layer_outputs = [layer.output for layer in model.layers if 'conv' in layer.name]
activation_model = Model(inputs=model.input, outputs=layer_outputs)

img = X_flat_gauss[0:1]  # one sample, shape (1, 69, 69, 1)
activations = activation_model.predict(img)

first_layer_activation = activations[0]  # (1, 69, 69, 8)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(first_layer_activation[0, :, :, i], cmap='viridis')
    ax.axis('off')
# plt.title("1st layer feature map - no smooth")
plt.savefig("plots/1LayerFM_no_smooth.png")
plt.show()

In [46]:
# 2. Grab every conv layer's output
conv_layers = [layer for layer in model.layers if 'conv' in layer.name]
layer_outputs = [layer.output for layer in conv_layers]
layer_names = [layer.name for layer in conv_layers]

activation_model = Model(inputs=model.input, outputs=layer_outputs)
activations = activation_model.predict(img)

# 3. Plot ONLY the second conv layer (index 1)
layer_name = layer_names[1]
activation = activations[1]

n_channels = activation.shape[-1]
n_cols = 8
n_rows = int(np.ceil(n_channels / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.5, n_rows * 1.5))
fig.suptitle(f"Layer: {layer_name}  (shape {activation.shape[1:]})", fontsize=12)

for i, ax in enumerate(axes.flat):
    if i < n_channels:
        ax.imshow(activation[0, :, :, i], cmap='viridis')
    ax.axis('off')

plt.tight_layout()
# plt.title("2nd layer feature map - no smooth")
plt.savefig("plots/2LayerFM_no_smooth.pdf")
plt.show()

In [47]:
layer_outputs = [layer.output for layer in gauss_model.layers if 'conv' in layer.name]
activation_model = Model(inputs=gauss_model.input, outputs=layer_outputs)

img = X_flat_gauss[0:1]  # one sample, shape (1, 69, 69, 1)
activations = activation_model.predict(img)

first_layer_activation = activations[0]  # (1, 69, 69, 8)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    ax.imshow(first_layer_activation[0, :, :, i], cmap='viridis')
    ax.axis('off')
# plt.title("1st layer feature map - gauss smooth")
plt.savefig("plots/1LayerFM_gauss_smooth.png")
plt.show()

In [48]:
# 2. Grab every conv layer's output
conv_layers = [layer for layer in gauss_model.layers if 'conv' in layer.name]
layer_outputs = [layer.output for layer in conv_layers]
layer_names = [layer.name for layer in conv_layers]

activation_model = Model(inputs=gauss_model.input, outputs=layer_outputs)
activations = activation_model.predict(img)

# 3. Plot ONLY the second conv layer (index 1)
layer_name = layer_names[1]
activation = activations[1]

n_channels = activation.shape[-1]
n_cols = 8
n_rows = int(np.ceil(n_channels / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 1.5, n_rows * 1.5))
fig.suptitle(f"Layer: {layer_name}  (shape {activation.shape[1:]})", fontsize=12)

for i, ax in enumerate(axes.flat):
    if i < n_channels:
        ax.imshow(activation[0, :, :, i], cmap='viridis')
    ax.axis('off')

plt.tight_layout()
# plt.title("2nd layer feature map - gauss smooth")
plt.savefig("plots/2LayerFM_gauss_smooth.pdf")
plt.show()